In [ ]:
## Cell 1 — 환경
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U transformers accelerate scipy

from google.colab import drive, userdata
drive.mount('/content/drive')

import torch, transformers, os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # bberylll 토큰, 시크릿에 등록
print(torch.__version__, transformers.__version__)

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 76.2 MB/s eta 0:00:00
Mounted at /content/drive
2.11.0+cu128 5.17.0


In [ ]:
## Cell 2 — 프롬프트 규격 + E0 러너
!wget -q -O p8s_prompt.py https://raw.githubusercontent.com/Ontology0/Graduation-Project/refs/heads/dev/prompt/p8s_prompt.py
# 위 URL은 실제 raw 경로 확인 후 수정

from p8s_prompt import PROMPT_VERSION, MODEL_ID, FEWSHOT, USER_TMPL, truncate_answer, score_row
print(PROMPT_VERSION, MODEL_ID)

p8s-v1 meta-llama/Llama-3.1-8B-Instruct


In [ ]:
## Cell 3 — 데이터 스키마 / 셀 균형 확인
import pandas as pd
P9 = "/content/drive/MyDrive/Grad_share/data_share/P9/"
df = pd.read_csv(P9 + "p8s_behavior.csv")

print(df.shape)
print(list(df.columns))
print(df.head(2).T)

# K/C 라벨 후보 컬럼의 분포
for c in df.columns:
    if df[c].nunique() <= 4 and df[c].dtype != float:
        print(c, df[c].value_counts().to_dict())

print("질문 수:", df["question"].nunique() if "question" in df else "컬럼명 확인 필요")

(1224, 19)
['question', 'gold', 'prop', 'kside', 'knows', 'A_cb', 'ctx_id', 'ctx_title', 's_wiki_title', 'orig_gold', 'ctx_label', 'ctx_text', 'answer_surface', 'ans_start', 'ans_end', 'wrong_answer', 'answer', 'behavior', 'final_correct']
                                                                0  \
question                                    What genre is Kanoon?   
gold            ["Bollywood", "HFI", "HLFI", "Hindi language f...   
prop                                                        genre   
kside                                                          K+   
knows                                                        True   
A_cb                            Kanoon is a Bollywood film genre.   
ctx_id                                                   16462170   
ctx_title                                         Farz Aur Kanoon   
s_wiki_title                                               Kanoon   
orig_gold                                               Bollywood   
c

In [ ]:
## Cell 4 — 기존 behavior로 헤드룸 확인
import pandas as pd
KM, CM = "K−", "C-"

ct = pd.crosstab([df.kside, df.ctx_label], df.behavior, normalize="index").round(3)
print(ct)
print("\nC- 행만:", df[df.ctx_label==CM].behavior.value_counts(normalize=True).round(3).to_dict())

behavior         abstain  accept  other  retain
kside ctx_label                                
K+    C+           0.000   0.762  0.079   0.159
      C-           0.003   0.593  0.127   0.278
K−    C+           0.000   0.769  0.132   0.098
      C-           0.000   0.803  0.141   0.056

C- 행만: {'accept': 0.673, 'retain': 0.193, 'other': 0.132, 'abstain': 0.002}


In [ ]:
## Cell 5
import json, numpy as np

KP, KM = "K+", "K−"
CP, CM = "C+", "C-"
COLS = dict(q="question", ctx="ctx_text", gold="gold",
            surf="answer_surface", k="kside", c="ctx_label")

df["gold_list"] = df["gold"].apply(json.loads)      # eval 말고 json (쌍따옴표 배열)
df["cell"] = df[COLS["k"]] + df[COLS["c"]]

rng = np.random.RandomState(42)
qs = np.sort(df.question.unique()); rng.shuffle(qs)
half = set(qs[:len(qs)//2])
df["split"] = np.where(df.question.isin(half), "A", "B")   # 재현성 태그용, 분석은 전량

# C− 전량 + C+ 200
sample = pd.concat([
    df[df.ctx_label == CM],
    df[df.ctx_label == CP].sample(200, random_state=42),
]).reset_index(drop=True)
sample["row_uid"] = np.arange(len(sample))

print(len(sample), sample.cell.value_counts().to_dict())

812 {'K+C-': 378, 'K−C-': 234, 'K+C+': 131, 'K−C+': 69}


In [ ]:
## Cell 6 — arm 정의 + 모델 로드
import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM

FMT = ("Output the answer string itself and nothing else: no sentence, no explanation, "
       "no punctuation, no quotes. Keep it as short as possible (a name or a term).")
NO_ABSTAIN = "If the context does not state the answer, output the most likely short answer anyway."
BASE = "You are a question answering system. "

ARMS = {
  "A0_only":      dict(sys=BASE+f"Answer the question using ONLY the given context. {FMT} {NO_ABSTAIN}", fs=True),
  "A1_based":     dict(sys=BASE+f"Answer the question based on the given context. {FMT} {NO_ABSTAIN}", fs=True),
  "A2_noabstain": dict(sys=BASE+f"Answer the question based on the given context. {FMT}",             fs=True),
  "A3_nofewshot": dict(sys=BASE+f"Answer the question based on the given context. {FMT} {NO_ABSTAIN}", fs=False),
}

def build_msgs(q, ctx, arm):
    m = [{"role":"system","content":ARMS[arm]["sys"]}]
    if ARMS[arm]["fs"]:
        for ex in FEWSHOT:
            m += [{"role":"user","content":USER_TMPL.format(ctx=ex["context"], q=ex["question"])},
                  {"role":"assistant","content":ex["answer"]}]
    m.append({"role":"user","content":USER_TMPL.format(ctx=ctx, q=q)})
    return m

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
EOS_IDS = {tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} - {None}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
    attn_implementation="sdpa", token=os.environ["HF_TOKEN"]).eval()
print(model.dtype, next(model.parameters()).device)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

torch.bfloat16 cuda:0


In [ ]:
## Cell 6d — BS 확정 (최장 프롬프트로 미리 터뜨려보기)
BS = 16
longest = sorted(range(len(sample)), key=lambda i: len(sample.ctx_text.iloc[i]))[-BS:]
enc = tok([tok.apply_chat_template(build_msgs(sample.question.iloc[i], sample.ctx_text.iloc[i], "A0_only"),
           tokenize=False, add_generation_prompt=True) for i in longest],
          return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
with torch.no_grad():
    _ = model.generate(**enc, max_new_tokens=192, do_sample=False,
                       pad_token_id=tok.pad_token_id, eos_token_id=list(EOS_IDS))
print("OK", enc["input_ids"].shape, f"{torch.cuda.max_memory_allocated()/1e9:.1f}GB")
torch.cuda.reset_peak_memory_stats()

OK torch.Size([16, 622]) 18.6GB


In [ ]:
PROMPT_MD5 = "p8s-v1 meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()


In [ ]:
import hashlib, transformers
PROMPT_MD5 = hashlib.md5(open("/content/p8s_prompt.py","rb").read()).hexdigest()
print(PROMPT_MD5, transformers.__version__)

88719f2ac436a1f3418021eedbb7f084 5.17.0


In [ ]:
print(PROMPT_MD5)

88719f2ac436a1f3418021eedbb7f084


In [ ]:
## Cell 7 — 생성 (배치 단위 자동 저장 + resume)
import numpy as np, pandas as pd, os, time, json

MAX_NEW, BS = 192, 16
DECODE_KW = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)
OUT = P9 + "E0/"; os.makedirs(OUT, exist_ok=True)

@torch.no_grad()
def run_arm(arm):
    done_f = OUT + f"e0_{arm}_partial.jsonl"
    final_f = OUT + f"e0_{arm}.parquet"
    if os.path.exists(final_f):
        print(f"skip {arm}"); return pd.read_parquet(final_f)

    done = {}
    if os.path.exists(done_f):
        for ln in open(done_f, encoding="utf-8"):
            d = json.loads(ln); done[d["row_uid"]] = d
        print(f"{arm}: resume {len(done)}/{len(sample)}")

    prompts = [tok.apply_chat_template(build_msgs(r.question, r.ctx_text, arm),
               tokenize=False, add_generation_prompt=True) for r in sample.itertuples()]
    L = [len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
    order = [i for i in np.argsort(L, kind="stable")
             if int(sample.row_uid.iloc[i]) not in done]

    t0 = time.time()
    with open(done_f, "a", encoding="utf-8") as fh:
        for s in range(0, len(order), BS):
            idx = order[s:s+BS]
            enc = tok([prompts[i] for i in idx], return_tensors="pt",
                      padding=True, add_special_tokens=False).to(model.device)
            gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                 pad_token_id=tok.pad_token_id,
                                 eos_token_id=list(EOS_IDS), use_cache=True
                                )[:, enc["input_ids"].shape[1]:]
            for j, i in enumerate(idx):
                g = gen[j].tolist()
                stop = next((p for p, t in enumerate(g) if t in EOS_IDS), None)
                n = stop if stop is not None else len(g)
                fh.write(json.dumps({"row_uid": int(sample.row_uid.iloc[i]),
                                     "raw_output": tok.decode(g[:n], **DECODE_KW),
                                     "n_tok": n, "hit_limit": int(stop is None)},
                                    ensure_ascii=False) + "\n")
            fh.flush(); os.fsync(fh.fileno())          # Drive에 즉시 기록
            if s % (BS*10) == 0:
                print(f"  {arm} {s+len(idx)}/{len(order)} {time.time()-t0:.0f}s", flush=True)

    recs = {}
    for ln in open(done_f, encoding="utf-8"):
        d = json.loads(ln); recs[d["row_uid"]] = d
    g = pd.DataFrame([recs[u] for u in sample.row_uid])
    out = sample.copy()
    for c in ["raw_output", "n_tok", "hit_limit"]: out[c] = g[c].values
    out["arm"], out["sys_prompt"] = arm, ARMS[arm]["sys"]
    out["prompt_md5"], out["prompt_version"] = PROMPT_MD5, PROMPT_VERSION
    out["tf_version"], out["bs"], out["max_new"] = transformers.__version__, BS, MAX_NEW
    out.to_parquet(final_f, index=False)
    print(f"{arm} DONE {time.time()-t0:.0f}s  hit_limit={out.hit_limit.mean():.3f}  "
          f"mean_tok={out.n_tok.mean():.1f}")
    return out

res = pd.concat([run_arm(a) for a in ARMS], ignore_index=True)
res.to_parquet(OUT + "e0_all.parquet", index=False)
print(res.groupby("arm").agg(hit_limit=("hit_limit","mean"), mean_tok=("n_tok","mean")).round(3))

A0_only: resume 812/812
A0_only DONE 0s  hit_limit=0.000  mean_tok=3.3
  A1_based 16/812 3s
  A1_based 176/812 36s
  A1_based 336/812 67s
  A1_based 496/812 103s
  A1_based 656/812 138s
  A1_based 812/812 174s
A1_based DONE 174s  hit_limit=0.000  mean_tok=3.2
  A2_noabstain 16/812 4s
  A2_noabstain 176/812 36s
  A2_noabstain 336/812 66s
  A2_noabstain 496/812 101s
  A2_noabstain 656/812 136s
  A2_noabstain 812/812 172s
A2_noabstain DONE 172s  hit_limit=0.000  mean_tok=3.3
  A3_nofewshot 16/812 2s
  A3_nofewshot 176/812 19s
  A3_nofewshot 336/812 37s
  A3_nofewshot 496/812 57s
  A3_nofewshot 656/812 79s
  A3_nofewshot 812/812 103s
A3_nofewshot DONE 103s  hit_limit=0.000  mean_tok=3.2
              hit_limit  mean_tok
arm                              
A0_only             0.0     3.260
A1_based            0.0     3.169
A2_noabstain        0.0     3.288
A3_nofewshot        0.0     3.213


In [ ]:
## Cell 8 — 채점
import numpy as np, pandas as pd, re
from p8s_prompt import truncate_answer, score_row

ABSTAIN_RE = re.compile(
    r"(i (don'?t|do not) know|unknown|not (stated|mentioned|provided|specified|given)|"
    r"cannot (be )?determin|no answer|insufficient|^n/?a$)", re.I)

res = pd.read_parquet(OUT + "e0_all.parquet")
res["pred"] = res.raw_output.map(truncate_answer)
sc = res.apply(lambda r: score_row(r.pred, r.gold_list, r.answer_surface), axis=1)
res[["em", "sub", "follows_ctx"]] = pd.DataFrame(sc.tolist(), index=res.index)
res["abstain"] = res.raw_output.str.contains(ABSTAIN_RE).astype(int)
res["n_words"] = res.pred.str.split().str.len()
res.to_parquet(OUT + "e0_scored.parquet", index=False)

piv = res.groupby(["arm", "cell"]).agg(
    n=("em", "size"), accept=("follows_ctx", "mean"), em=("em", "mean"),
    abstain=("abstain", "mean"), over6=("n_words", lambda x: (x > 6).mean())).round(3)
print(piv.to_string())

# K+C- vs K-C- accept 격차 (실험 생존의 핵심 지표)
a = res[res.ctx_label == CM].groupby(["arm", "kside"]).follows_ctx.mean().unstack()
a["gap"] = (a["K−"] - a["K+"]).round(3)
print("\n", a.round(3).to_string())

                     n  accept     em  abstain  over6
arm          cell                                    
A0_only      K+C+  131   0.687  0.863    0.015  0.000
             K+C-  378   0.595  0.106    0.013  0.003
             K−C+   69   0.681  0.710    0.014  0.000
             K−C-  234   0.714  0.026    0.030  0.013
A1_based     K+C+  131   0.702  0.870    0.008  0.000
             K+C-  378   0.503  0.214    0.013  0.000
             K−C+   69   0.681  0.710    0.014  0.000
             K−C-  234   0.697  0.051    0.021  0.009
A2_noabstain K+C+  131   0.702  0.870    0.008  0.008
             K+C-  378   0.503  0.212    0.016  0.013
             K−C+   69   0.681  0.710    0.000  0.000
             K−C-  234   0.705  0.056    0.017  0.009
A3_nofewshot K+C+  131   0.695  0.870    0.000  0.008
             K+C-  378   0.394  0.373    0.003  0.003
             K−C+   69   0.696  0.710    0.000  0.000
             K−C-  234   0.684  0.060    0.000  0.000

 kside            K+     K−

/tmp/ipykernel_1178/365883600.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  res["abstain"] = res.raw_output.str.contains(ABSTAIN_RE).astype(int)


In [ ]:
## Cell 9 — A5 추가 (A3 - abstain문장) + regex 경고 수정
ARMS["A5_final"] = dict(sys=BASE+f"Answer the question based on the given context. {FMT}", fs=False)
res5 = run_arm("A5_final")

ABSTAIN_RE = re.compile(
    r"(?:i (?:don'?t|do not) know|unknown|not (?:stated|mentioned|provided|specified|given)|"
    r"cannot (?:be )?determin|no answer|insufficient|^n/?a$)", re.I)

  A5_final 16/812 2s
  A5_final 176/812 19s
  A5_final 336/812 36s
  A5_final 496/812 55s
  A5_final 656/812 75s
  A5_final 812/812 96s
A5_final DONE 96s  hit_limit=0.000  mean_tok=3.2


In [ ]:
## Cell 8b — A5 포함 재채점
res = pd.concat([pd.read_parquet(OUT+f"e0_{a}.parquet") for a in ARMS], ignore_index=True)
res["pred"] = res.raw_output.map(truncate_answer)
sc = res.apply(lambda r: score_row(r.pred, r.gold_list, r.answer_surface), axis=1)
res[["em","sub","follows_ctx"]] = pd.DataFrame(sc.tolist(), index=res.index)
res["abstain"] = res.raw_output.str.contains(ABSTAIN_RE).astype(int)
res["n_words"] = res.pred.str.split().str.len()
res.to_parquet(OUT+"e0_scored.parquet", index=False)

full = res
print(pd.concat([mcnemar(full,"follows_ctx","K+C-"),
                 mcnemar(full,"em","K+C-")]).query("a=='A3_nofewshot' or b=='A3_nofewshot'").to_string(index=False))

cell      metric            a            b  rate_a  rate_b  n01  n10        p
K+C- follows_ctx      A0_only A3_nofewshot   0.595   0.394    8   84 4.14e-17
K+C- follows_ctx     A1_based A3_nofewshot   0.503   0.394   13   54 4.47e-07
K+C- follows_ctx A2_noabstain A3_nofewshot   0.503   0.394   11   52 1.67e-07
K+C- follows_ctx A3_nofewshot     A5_final   0.394   0.405    9    5 4.24e-01
K+C-          em      A0_only A3_nofewshot   0.106   0.373  105    4 1.78e-26
K+C-          em     A1_based A3_nofewshot   0.214   0.373   67    7 2.12e-13
K+C-          em A2_noabstain A3_nofewshot   0.212   0.373   69    8 3.14e-13
K+C-          em A3_nofewshot     A5_final   0.373   0.370    8    9 1.00e+00


In [ ]:
## Cell 10 — McNemar (paired)
from scipy.stats import binomtest
import itertools

def mcnemar(df, metric, cell):
    d = df[df.cell == cell].pivot(index="row_uid", columns="arm", values=metric)
    out = []
    for a, b in itertools.combinations(d.columns, 2):
        n01 = int(((d[a]==0)&(d[b]==1)).sum()); n10 = int(((d[a]==1)&(d[b]==0)).sum())
        p = binomtest(n01, n01+n10, 0.5).pvalue if n01+n10 else 1.0
        out.append(dict(cell=cell, metric=metric, a=a, b=b,
                        rate_a=round(d[a].mean(),3), rate_b=round(d[b].mean(),3),
                        n01=n01, n10=n10, p=f"{p:.2e}"))
    return pd.DataFrame(out)

full = pd.read_parquet(OUT+"e0_scored.parquet")   # A5 포함해 재채점 후
print(pd.concat([mcnemar(full,"follows_ctx","K+C-"),
                 mcnemar(full,"em","K+C-"),
                 mcnemar(full,"follows_ctx","K−C-")]).to_string(index=False))

cell      metric            a            b  rate_a  rate_b  n01  n10        p
K+C- follows_ctx      A0_only     A1_based   0.595   0.503    1   36 5.53e-10
K+C- follows_ctx      A0_only A2_noabstain   0.595   0.503    0   35 5.82e-11
K+C- follows_ctx      A0_only A3_nofewshot   0.595   0.394    8   84 4.14e-17
K+C- follows_ctx     A1_based A2_noabstain   0.503   0.503    7    7 1.00e+00
K+C- follows_ctx     A1_based A3_nofewshot   0.503   0.394   13   54 4.47e-07
K+C- follows_ctx A2_noabstain A3_nofewshot   0.503   0.394   11   52 1.67e-07
K+C-          em      A0_only     A1_based   0.106   0.214   41    0 9.09e-13
K+C-          em      A0_only A2_noabstain   0.106   0.212   41    1 1.96e-11
K+C-          em      A0_only A3_nofewshot   0.106   0.373  105    4 1.78e-26
K+C-          em     A1_based A2_noabstain   0.214   0.212    6    7 1.00e+00
K+C-          em     A1_based A3_nofewshot   0.214   0.373   67    7 2.12e-13
K+C-          em A2_noabstain A3_nofewshot   0.212   0.373   69 

In [ ]:
## Cell 11 — 불일치 쌍만 사람 검증용 추출
d = full[(full.cell=="K+C-") & full.arm.isin(["A0_only","A3_nofewshot"])] \
        .pivot(index="row_uid", columns="arm", values="follows_ctx")
disc = d[d.A0_only != d.A3_nofewshot].index
aud = full[full.row_uid.isin(disc) & full.arm.isin(["A0_only","A3_nofewshot"])] \
        [["row_uid","arm","question","answer_surface","gold","raw_output","pred","follows_ctx","em"]]
aud.sort_values(["row_uid","arm"]).to_csv(OUT+"e0_audit.csv", index=False)
print(len(disc), "쌍")

92 쌍


In [ ]:
## Cell 11b
d = full[(full.cell=="K+C-") & full.arm.isin(["A0_only","A5_final"])] \
        .pivot(index="row_uid", columns="arm", values="follows_ctx")
disc = d[d.A0_only != d.A5_final].index
full[full.row_uid.isin(disc) & full.arm.isin(["A0_only","A5_final"])] \
    [["row_uid","arm","question","gold","answer_surface","raw_output","pred","follows_ctx","em"]] \
    .sort_values(["row_uid","arm"]).to_csv(OUT+"e0_audit.csv", index=False)
print(len(disc))

96


In [ ]:
aud = pd.read_csv(OUT+"e0_audit.csv")
w = aud.pivot(index="row_uid", columns="arm", values=["raw_output","follows_ctx"]).reset_index()
w.columns = ["row_uid","out_A0","out_A5","fc_A0","fc_A5"]
w = w.merge(aud[["row_uid","question","gold","answer_surface"]].drop_duplicates("row_uid"), on="row_uid")
w["label_A0"] = ""; w["label_A5"] = ""
w.to_csv(OUT+"e0_audit_sheet.csv", index=False)
print(len(w))

96


### 2차 작업

In [ ]:
## Cell A — 최소 재시작 (GPU 불필요)
from google.colab import drive; drive.mount('/content/drive')
import pandas as pd, numpy as np, re, json
from scipy.stats import binomtest

P9  = "/content/drive/MyDrive/Grad_share/data_share/P9/"
OUT = P9 + "E0/"
ARM_NAMES = ["A0_only","A1_based","A2_noabstain","A3_nofewshot","A5_final"]

full = pd.concat([pd.read_parquet(OUT+f"e0_{a}.parquet") for a in ARM_NAMES], ignore_index=True)
print(full.shape, full.arm.unique())

Mounted at /content/drive
(4060, 33) ['A0_only' 'A1_based' 'A2_noabstain' 'A3_nofewshot' 'A5_final']


In [ ]:
import os; print(os.listdir(OUT))

['e0_A0_only_partial.jsonl', 'e0_A0_only.parquet', 'e0_A1_based_partial.jsonl', 'e0_A1_based.parquet', 'e0_A2_noabstain_partial.jsonl', 'e0_A2_noabstain.parquet', 'e0_A3_nofewshot_partial.jsonl', 'e0_A3_nofewshot.parquet', 'e0_all.parquet', 'e0_A5_final_partial.jsonl', 'e0_A5_final.parquet', 'e0_scored.parquet', 'e0_audit.csv', 'e0_audit_sheet.csv', 'e0_audit_sheet.gsheet', 'e0_audit_sheet_labeled.csv']


In [ ]:
full = pd.read_parquet(OUT+"e0_scored.parquet")

In [ ]:
print(full.arm.unique(), full.shape)

['A0_only' 'A1_based' 'A2_noabstain' 'A3_nofewshot' 'A5_final'] (4060, 39)


In [ ]:
## Cell B — 최종 집계
lab = pd.read_csv(OUT+"e0_audit_sheet_labeled.csv")
print(pd.crosstab(lab.label_A0, lab.label_A5, margins=True))

k = full[(full.cell=="K+C-")]
N = k.arm.value_counts().iloc[0]
a0 = k[k.arm=="A0_only"]
const = int(a0.follows_ctx.sum()) - int(a0[a0.row_uid.isin(lab.row_uid)].follows_ctx.sum())

n10 = int(((lab.label_A0=="CTX") & (lab.label_A5!="CTX")).sum())
n01 = int(((lab.label_A0!="CTX") & (lab.label_A5=="CTX")).sum())
tie = int(((lab.label_A0=="CTX") & (lab.label_A5=="CTX")).sum())
gold = int(((lab.label_A0=="CTX") & (lab.label_A5=="GOLD")).sum())

print(f"N={N}  human A0={(const+n10+tie)/N:.3f}  A5={(const+n01+tie)/N:.3f}  Δ={(n10-n01)/N:.3f}")
print(f"accept 감소 중 GOLD 전환: {gold}/{n10} = {gold/n10:.3f}")
print("McNemar p =", binomtest(n01, n01+n10, 0.5).pvalue)
print("\nflag:"); print(pd.concat([lab.flag_A0, lab.flag_A5]).dropna().value_counts())

label_A5  CTX  GOLD  NONE  All
label_A0                      
CTX         2    72    11   85
GOLD        6     0     0    6
NONE        2     0     3    5
All        10    72    14   96
N=378  human A0=0.598  A5=0.399  Δ=0.198
accept 감소 중 GOLD 전환: 72/83 = 0.867
McNemar p = 7.576841438713428e-17

flag:
partial_string    8
gold_defect       4
granularity       3
multi             3
surname           2
abstain           1
Name: count, dtype: int64


### 3차

In [ ]:
## Cell 1 — 환경
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U transformers accelerate scipy

from google.colab import drive, userdata
drive.mount('/content/drive')
import os, torch, transformers
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print(torch.__version__, transformers.__version__)

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 72.8 MB/s eta 0:00:00
Mounted at /content/drive
2.11.0+cu128 5.17.0


In [ ]:
## Cell 2 — 프롬프트 파일 + 경로
import shutil, hashlib, json, re, time
import numpy as np, pandas as pd

!wget -q -O p8s_prompt.py https://raw.githubusercontent.com/Ontology0/Graduation-Project/refs/heads/dev/prompt/p8s_prompt.py

txt = open("/content/p8s_prompt.py", encoding="utf-8").read()
assert "<title>" not in txt and "PROMPT_VERSION" in txt, "HTML을 받았음 — URL 확인"

PROMPT_MD5 = hashlib.md5(txt.encode("utf-8")).hexdigest()
from p8s_prompt import PROMPT_VERSION, MODEL_ID, FEWSHOT, USER_TMPL, truncate_answer, score_row

P9  = "/content/drive/MyDrive/Grad_share/data_share/P9/"
OUT = P9 + "E0/"
print(PROMPT_MD5, PROMPT_VERSION)
assert PROMPT_MD5 == "88719f2ac436a1f3418021eedbb7f084", "이전 arm과 프롬프트 파일이 다름"
print(sorted(os.listdir(OUT)))

88719f2ac436a1f3418021eedbb7f084 p8s-v1
['e0_A0_only.parquet', 'e0_A0_only_partial.jsonl', 'e0_A1_based.parquet', 'e0_A1_based_partial.jsonl', 'e0_A2_noabstain.parquet', 'e0_A2_noabstain_partial.jsonl', 'e0_A3_nofewshot.parquet', 'e0_A3_nofewshot_partial.jsonl', 'e0_A5_final.parquet', 'e0_A5_final_partial.jsonl', 'e0_all.parquet', 'e0_audit.csv', 'e0_audit_sheet.csv', 'e0_audit_sheet.gsheet', 'e0_audit_sheet_labeled.csv', 'e0_scored.parquet']


In [ ]:
## Cell 3 — sample 재생성 (seed 고정이라 이전과 동일한 row_uid)
KP, KM, CP, CM = "K+", "K−", "C+", "C-"
df = pd.read_csv(P9 + "p8s_behavior.csv")
df["gold_list"] = df["gold"].apply(json.loads)
df["cell"] = df["kside"] + df["ctx_label"]

sample = pd.concat([
    df[df.ctx_label == CM],
    df[df.ctx_label == CP].sample(200, random_state=42),
]).reset_index(drop=True)
sample["row_uid"] = np.arange(len(sample))
print(len(sample), sample.cell.value_counts().to_dict())

812 {'K+C-': 378, 'K−C-': 234, 'K+C+': 131, 'K−C+': 69}


In [ ]:
## Cell 4 — arm 정의 + 모델 로드
FMT = ("Output the answer string itself and nothing else: no sentence, no explanation, "
       "no punctuation, no quotes. Keep it as short as possible (a name or a term).")
BASE = "You are a question answering system. "

ARMS = {
  "A6_noinstr": dict(sys=BASE + FMT, fs=False),
  "A7_fmtonly": dict(sys=FMT,        fs=False),
  "A8_prior":   dict(sys=BASE + "According to the given information and your knowledge, "
                               "answer the question. " + FMT, fs=False),
}

def build_msgs(q, ctx, arm):
    m = [{"role":"system","content":ARMS[arm]["sys"]}]
    if ARMS[arm]["fs"]:
        for ex in FEWSHOT:
            m += [{"role":"user","content":USER_TMPL.format(ctx=ex["context"], q=ex["question"])},
                  {"role":"assistant","content":ex["answer"]}]
    m.append({"role":"user","content":USER_TMPL.format(ctx=ctx, q=q)})
    return m

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
EOS_IDS = {tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} - {None}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
    attn_implementation="sdpa", token=os.environ["HF_TOKEN"]).eval()
print(model.dtype, next(model.parameters()).device)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

torch.bfloat16 cuda:0


In [ ]:
## Cell 5 — 생성 (BS=16 고정, 이전 arm과 동일 조건)
MAX_NEW, BS = 192, 16
DECODE_KW = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)

@torch.no_grad()
def run_arm(arm):
    final_f = OUT + f"e0_{arm}.parquet"
    if os.path.exists(final_f):
        print(f"skip {arm}"); return pd.read_parquet(final_f)
    done_f = OUT + f"e0_{arm}_partial.jsonl"
    done = {}
    if os.path.exists(done_f):
        for ln in open(done_f, encoding="utf-8"):
            d = json.loads(ln); done[d["row_uid"]] = d
        print(f"{arm}: resume {len(done)}/{len(sample)}")

    prompts = [tok.apply_chat_template(build_msgs(r.question, r.ctx_text, arm),
               tokenize=False, add_generation_prompt=True) for r in sample.itertuples()]
    L = [len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
    order = [i for i in np.argsort(L, kind="stable")
             if int(sample.row_uid.iloc[i]) not in done]

    t0 = time.time()
    with open(done_f, "a", encoding="utf-8") as fh:
        for s in range(0, len(order), BS):
            idx = order[s:s+BS]
            enc = tok([prompts[i] for i in idx], return_tensors="pt",
                      padding=True, add_special_tokens=False).to(model.device)
            gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                 pad_token_id=tok.pad_token_id,
                                 eos_token_id=list(EOS_IDS), use_cache=True
                                )[:, enc["input_ids"].shape[1]:]
            for j, i in enumerate(idx):
                g = gen[j].tolist()
                stop = next((p for p, t in enumerate(g) if t in EOS_IDS), None)
                n = stop if stop is not None else len(g)
                fh.write(json.dumps({"row_uid": int(sample.row_uid.iloc[i]),
                                     "raw_output": tok.decode(g[:n], **DECODE_KW),
                                     "n_tok": n, "hit_limit": int(stop is None)},
                                    ensure_ascii=False) + "\n")
            fh.flush(); os.fsync(fh.fileno())
            if s % (BS*10) == 0: print(f"  {arm} {s+len(idx)}/{len(order)} {time.time()-t0:.0f}s", flush=True)

    recs = {}
    for ln in open(done_f, encoding="utf-8"):
        d = json.loads(ln); recs[d["row_uid"]] = d
    g = pd.DataFrame([recs[u] for u in sample.row_uid])
    out = sample.copy()
    for c in ["raw_output","n_tok","hit_limit"]: out[c] = g[c].values
    out["arm"], out["sys_prompt"] = arm, ARMS[arm]["sys"]
    out["prompt_md5"], out["prompt_version"] = PROMPT_MD5, PROMPT_VERSION
    out["tf_version"], out["bs"], out["max_new"] = transformers.__version__, BS, MAX_NEW
    out.to_parquet(final_f, index=False)
    print(f"{arm} DONE {time.time()-t0:.0f}s hit_limit={out.hit_limit.mean():.3f} mean_tok={out.n_tok.mean():.1f}")
    return out

for a in ARMS: run_arm(a)

  A6_noinstr 16/812 2s
  A6_noinstr 176/812 18s
  A6_noinstr 336/812 33s
  A6_noinstr 496/812 50s
  A6_noinstr 656/812 69s
  A6_noinstr 812/812 88s
A6_noinstr DONE 88s hit_limit=0.000 mean_tok=3.2
  A7_fmtonly 16/812 1s
  A7_fmtonly 176/812 17s
  A7_fmtonly 336/812 32s
  A7_fmtonly 496/812 50s
  A7_fmtonly 656/812 68s
  A7_fmtonly 812/812 88s
A7_fmtonly DONE 88s hit_limit=0.000 mean_tok=3.3
  A8_prior 16/812 1s
  A8_prior 176/812 18s
  A8_prior 336/812 35s
  A8_prior 496/812 54s
  A8_prior 656/812 74s
  A8_prior 812/812 95s
A8_prior DONE 95s hit_limit=0.000 mean_tok=3.2


In [ ]:
## Cell 6 — 재채점 + A5↔A6↔A7 비교
from scipy.stats import binomtest
import itertools

ARM_ALL = ["A0_only","A1_based","A2_noabstain","A3_nofewshot",
           "A5_final","A6_noinstr","A7_fmtonly","A8_prior"]
ABSTAIN_RE = re.compile(
    r"(?:i (?:don'?t|do not) know|unknown|not (?:stated|mentioned|provided|specified|given)|"
    r"cannot (?:be )?determin|no answer|insufficient|^n/?a$)", re.I)

full = pd.concat([pd.read_parquet(OUT+f"e0_{a}.parquet") for a in ARM_ALL], ignore_index=True)
if "gold_list" not in full.columns: full["gold_list"] = full.gold.apply(json.loads)
full["pred"] = full.raw_output.map(truncate_answer)
sc = full.apply(lambda r: score_row(r.pred, r.gold_list, r.answer_surface), axis=1)
full[["em","sub","follows_ctx"]] = pd.DataFrame(sc.tolist(), index=full.index)
full["n_words"] = full.pred.str.split().str.len()
full["abstain"] = full.raw_output.str.contains(ABSTAIN_RE).astype(int)
full.to_parquet(OUT+"e0_scored.parquet", index=False)

print(full[full.cell=="K+C-"].groupby("arm").agg(
    accept=("follows_ctx","mean"), em=("em","mean"),
    mean_tok=("n_tok","mean"), over6=("n_words", lambda x:(x>6).mean())).round(3).to_string())

def mcnemar(df, metric, cell, arms):
    d = df[(df.cell==cell) & df.arm.isin(arms)].pivot(index="row_uid", columns="arm", values=metric)
    r=[]
    for a,b in itertools.combinations(d.columns,2):
        n01=int(((d[a]==0)&(d[b]==1)).sum()); n10=int(((d[a]==1)&(d[b]==0)).sum())
        p=binomtest(n01,n01+n10,0.5).pvalue if n01+n10 else 1.0
        r.append(dict(a=a,b=b,rate_a=round(d[a].mean(),3),rate_b=round(d[b].mean(),3),
                      n01=n01,n10=n10,p=f"{p:.2e}"))
    return pd.DataFrame(r)

print("\n", mcnemar(full,"follows_ctx","K+C-",
                    ["A5_final","A6_noinstr","A7_fmtonly","A8_prior"]).to_string(index=False))

              accept     em  mean_tok  over6
arm                                         
A0_only        0.595  0.106     3.101  0.003
A1_based       0.503  0.214     2.997  0.000
A2_noabstain   0.503  0.212     3.201  0.013
A3_nofewshot   0.394  0.373     3.061  0.003
A5_final       0.405  0.370     3.053  0.000
A6_noinstr     0.317  0.471     2.989  0.000
A7_fmtonly     0.389  0.381     3.085  0.003
A8_prior       0.376  0.402     3.034  0.000

          a          b  rate_a  rate_b  n01  n10        p
  A5_final A6_noinstr   0.405   0.317    0   33 2.33e-10
  A5_final A7_fmtonly   0.405   0.389    6   12 2.38e-01
  A5_final   A8_prior   0.405   0.376    2   13 7.39e-03
A6_noinstr A7_fmtonly   0.317   0.389   30    3 1.40e-06
A6_noinstr   A8_prior   0.317   0.376   23    1 2.98e-06
A7_fmtonly   A8_prior   0.389   0.376    9   14 4.05e-01


In [ ]:
!wget -q -O p8s_prompt.py https://raw.githubusercontent.com/Ontology0/Graduation-Project/refs/heads/dev/prompt/p8s_prompt.py
import hashlib
print(hashlib.md5(open("p8s_prompt.py","rb").read()).hexdigest())

20959a4c34203ea0866d030baf23595d
